In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)


C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [3]:
df_pos = pd.read_csv("tashbeeh_dataset.csv")
df_neg = pd.read_csv("non_tashbeeh_random_sample.csv")

df_pos["label"] = 1
df_neg["label"] = 0

TEXT_COL = "البيت_الكامل"

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

# 🔥 REMOVE TASHKEEL
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

df.head()


,البيت_الكامل,label
0,أهم بشيء والليالي كأنها تطاردني عن كونه وأطارد,1
1,تثنى على قدر الطعان كأنما مفاصلها تحت الرماح م...,1
2,ولما رأيت الناس دون محله تيقنت أن الدهر للناس ...,1
3,مخضبة والقوم صرعى كأنها وإن لم يكونوا ساجدين م...,1
4,تنكسهم والسابقات جبالهم وتطعن فيهم والرماح الم...,1


In [4]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df[TEXT_COL].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)


In [5]:
MODEL_NAME = "riotu-lab/ArabianGPT-03B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

# ✅ GPT-style fix: set PAD token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# ✅ VERY IMPORTANT: sync model config
model.config.pad_token_id = tokenizer.pad_token_id


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more 

In [6]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=512
    )


In [7]:
train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
test_df  = pd.DataFrame({"text": test_texts,  "label": test_labels})


In [8]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer)


Map: 100%|██████████| 218/218 [00:00<00:00, 20417.99 examples/s]


In [9]:
training_args = TrainingArguments(
    output_dir="./arabianGPT_tashbeeh",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    # 🔥 KEEP ONLY THE BEST MODEL
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,   # 👈 deletes older checkpoints automatically

    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=4,

    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=10,

    fp16=True,
    report_to="none"
)


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\accelerate\accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [12]:
trainer.train()


  5%|▍         | 10/216 [00:14<04:47,  1.39s/it]

{'loss': 1.1958, 'learning_rate': 1.9351851851851853e-05, 'epoch': 0.18}


  9%|▉         | 20/216 [00:30<05:19,  1.63s/it]

{'loss': 0.8238, 'learning_rate': 1.8425925925925926e-05, 'epoch': 0.37}


 14%|█▍        | 30/216 [00:47<05:05,  1.64s/it]

...

{'eval_loss': 0.9116086363792419, 'eval_accuracy': 0.7339449541284404, 'eval_runtime': 2.528, 'eval_samples_per_second': 86.233, 'eval_steps_per_second': 21.756, 'epoch': 3.96}


100%|██████████| 216/216 [07:57<00:00,  1.61s/it]

{'train_runtime': 477.1397, 'train_samples_per_second': 7.31, 'train_steps_per_second': 0.453, 'train_loss': 0.40506459220691965, 'epoch': 3.96}


100%|██████████| 216/216 [07:57<00:00,  2.21s/it]


TrainOutput(global_step=216, training_loss=0.40506459220691965, metrics={'train_runtime': 477.1397, 'train_samples_per_second': 7.31, 'train_steps_per_second': 0.453, 'train_loss': 0.40506459220691965, 'epoch': 3.96})

In [13]:
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)

print("Accuracy:", accuracy_score(test_labels, y_pred))
print(confusion_matrix(test_labels, y_pred))
print(classification_report(test_labels, y_pred, target_names=["Non-Tashbeeh", "Tashbeeh"]))


100%|██████████| 55/55 [00:02<00:00, 24.14it/s]

Accuracy: 0.7385321100917431
[[84 25]
 [32 77]]
              precision    recall  f1-score   support

Non-Tashbeeh       0.72      0.77      0.75       109
    Tashbeeh       0.75      0.71      0.73       109

    accuracy                           0.74       218
   macro avg       0.74      0.74      0.74       218
weighted avg       0.74      0.74      0.74       218



In [14]:
trainer.save_model("arabianGPT-0.3B-tashbeeh")
tokenizer.save_pretrained("arabianGPT-0.3B-tashbeeh")


('arabianGPT-0.3B-tashbeeh\\tokenizer_config.json',
 'arabianGPT-0.3B-tashbeeh\\special_tokens_map.json',
 'arabianGPT-0.3B-tashbeeh\\tokenizer.json')